In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


In [ ]:
import squidpy as sq
import scanpy as sc
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:

adata = sc.read_h5ad("/data/beifen/zhongmin/slide-tag/H5AD格式数据/slide-tag肺_with_scanvi_加上补测数据_对3级淋巴结构进行分类_186_2025_11_10_4.h5ad")
print(adata)

In [ ]:
# 需要的包
import os, re
import numpy as np
import pandas as pd
import squidpy as sq
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

# ========= 可配置参数 =========
CLUSTER_KEY   = "celltype_3_ZZM"
SAMPLE_KEY    = "sample"
SUBTYPE_KEY   = "subtype"
OUT_DIR       = "/data/beifen/zhongmin/泛癌S/h5ad/计算空间距离/结果文件_2025_11_07"
COORD_TYPE    = "generic"
N_NEIGHS      = 6
N_PERMS       = 2000
SEED          = 0
MIN_CLASSES   = 2
MIN_CELLS_PER_CLASS = 5

os.makedirs(OUT_DIR, exist_ok=True)

def safe_filename(s: str) -> str:
    s = str(s) if s is not None else "NA"
    s = re.sub(r"[\\/:*?\"<>|]+", "_", s)
    s = re.sub(r"\s+", "_", s).strip("_")
    return s or "NA"

def ensure_spatial_in_obsm(ad):
    if "spatial" in ad.obsm and ad.obsm["spatial"] is not None:
        return
    if "X_spatial" in ad.obsm and ad.obsm["X_spatial"] is not None:
        ad.obsm["spatial"] = ad.obsm["X_spatial"]; return
    if {"xcoord","ycoord"}.issubset(ad.obs.columns):
        ad.obsm["spatial"] = ad.obs[["xcoord","ycoord"]].to_numpy(); return
    raise ValueError("找不到空间坐标（obsm['spatial'] / 'X_spatial' 或 obs['xcoord','ycoord']）。")

def get_subtype_suffix(ad):
    if SUBTYPE_KEY not in ad.obs.columns:
        return "NA"
    vals = ad.obs[SUBTYPE_KEY].dropna().astype(str).unique().tolist()
    return "NA" if len(vals)==0 else "+".join(sorted(vals))

all_samples = adata.obs[SAMPLE_KEY].astype(str).unique().tolist()
print(f"将逐个处理 {len(all_samples)} 个样本……")

for s in all_samples:
    print(f"\n=== 处理样本: {s} ===")
    ad = adata[adata.obs[SAMPLE_KEY].astype(str) == s].copy()
    if ad.n_obs == 0:
        continue

    try:
        ensure_spatial_in_obsm(ad)
    except Exception as e:
        print(f"  跳过：{s} 无空间坐标：{e}")
        continue

    if CLUSTER_KEY not in ad.obs.columns:
        print(f"  跳过：{s} 缺少 {CLUSTER_KEY}")
        continue

    # 过滤稀有类
    ad.obs[CLUSTER_KEY] = ad.obs[CLUSTER_KEY].astype("category")
    vc = ad.obs[CLUSTER_KEY].value_counts()
    kept = vc[vc >= MIN_CELLS_PER_CLASS].index.tolist()
    if len(kept) < MIN_CLASSES:
        print(f"  跳过：{s} 有效类别不足 {MIN_CLASSES}")
        continue
    ad = ad[ad.obs[CLUSTER_KEY].isin(kept)].copy()
    ad.obs[CLUSTER_KEY] = ad.obs[CLUSTER_KEY].astype("category")

    # 空间邻接
    if COORD_TYPE == "generic":
        sq.gr.spatial_neighbors(ad, coord_type="generic", n_neighs=N_NEIGHS)
    else:
        sq.gr.spatial_neighbors(ad, coord_type="grid", n_rings=1)

    # 邻域富集（置换）
    sq.gr.nhood_enrichment(ad, cluster_key=CLUSTER_KEY, n_perms=N_PERMS, seed=SEED)

    key = f"{CLUSTER_KEY}_nhood_enrichment"
    if key not in ad.uns or "zscore" not in ad.uns[key]:
        print(f"  警告：{s} 无 zscore 结果，跳过。"); continue

    # ---- 取 Z 矩阵并保存 ----
    cats = list(ad.obs[CLUSTER_KEY].cat.categories)
    Z = ad.uns[key]["zscore"]
    if not isinstance(Z, pd.DataFrame):
        Z = pd.DataFrame(np.asarray(Z), index=cats, columns=cats)

    subtype_suffix = get_subtype_suffix(ad)
    base = f"{safe_filename(s)}_{safe_filename(subtype_suffix)}"
    Z.to_csv(os.path.join(OUT_DIR, f"{base}.csv"))
    print(f"  保存 Z-score：{os.path.join(OUT_DIR, f'{base}.csv')}")

    # ---- 由 Z 计算双侧 p 值，并做 BH(FDR) 校正 ----
    p_mat = 2 * (1 - norm.cdf(np.abs(Z.values)))
    p_df  = pd.DataFrame(p_mat, index=Z.index, columns=Z.columns)

    # 多重校正（每个样本内一并校正）
    p_flat = p_df.values.ravel()
    q_flat = multipletests(p_flat, method="fdr_bh")[1]
    q_df   = pd.DataFrame(q_flat.reshape(p_df.shape), index=p_df.index, columns=p_df.columns)

    p_df.to_csv(os.path.join(OUT_DIR, f"{base}_pvalue.csv"))
    q_df.to_csv(os.path.join(OUT_DIR, f"{base}_qvalue_fdrbh.csv"))
    print(f"  保存 p 值：{os.path.join(OUT_DIR, f'{base}_pvalue.csv')}")
    print(f"  保存 q 值(FDR-BH)：{os.path.join(OUT_DIR, f'{base}_qvalue_fdrbh.csv')}")

    # （可选）保存 count 矩阵
    if "count" in ad.uns[key]:
        C = ad.uns[key]["count"]
        if not isinstance(C, pd.DataFrame):
            C = pd.DataFrame(np.asarray(C), index=cats, columns=cats)
        C.to_csv(os.path.join(OUT_DIR, f"{base}_count.csv"))
